<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>Cómo funcionan las redes neuronales</b></font></h1>
<h2><b>Anexo A: </b>PyTorch vs Numpy</h2>
<br>
Un tensor es una caja de números con forma y tipo, igual que un arreglo de NumPy. Acá van los
dos lado a lado, para ver dónde se parecen y dónde no.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

----

<br>

## **Parte 1:** Crear

La misma lista entra en los dos: **cuatro ejemplos, una característica cada uno**.

In [ ]:
lista = [[3.0], [7.0], [12.0], [18.0]]

print("torch.tensor(lista)")
print(torch.tensor(lista))

print("\nnp.array(lista)")
print(np.array(lista))

Las cajas prefabricadas también son iguales, pero **NumPy recibe las dimensiones en una tupla**.

In [ ]:
print("torch.zeros(2, 3)");       print(torch.zeros(2, 3))
print("\nnp.zeros((2, 3))");      print(np.zeros((2, 3)))

print("\ntorch.rand(2, 3)");      print(torch.rand(2, 3))
print("\nnp.random.rand(2, 3)");  print(np.random.rand(2, 3))

----

<br>

## **Parte 2:** La forma

`.shape` se lee igual en los dos: **(ejemplos, características)**.

In [ ]:
distancias = torch.tensor([[3.0], [7.0], [12.0], [18.0], [22.0], [28.0]])
distancias_np = distancias.numpy()

print("tensor:", tuple(distancias.shape), "| array:", distancias_np.shape)
print(distancias.shape[0], "ejemplos,", distancias.shape[1], "característica")

Ojo con `size`: en NumPy es cuántos números hay, en PyTorch es la forma.

In [ ]:
print("distancias.numel()  ->", distancias.numel())
print("distancias_np.size  ->", distancias_np.size)
print("distancias.size()   ->", distancias.size())

La forma importa porque la capa declara cuántas características espera por ejemplo.

In [ ]:
capa = nn.Linear(1, 1)   # espera 1 característica por ejemplo

print("entra", tuple(distancias.shape), "-> sale", tuple(capa(distancias).shape))

tres = torch.tensor([[3.0, 7.0, 1.0], [18.0, 22.0, 2.0]])   # 3 características
try:
    capa(tres)
except RuntimeError as error:
    print("\nRuntimeError:", error)

----

<br>

## **Parte 3:** El tipo

Con enteros coinciden. Con decimales no, y esa es la diferencia que más problemas causa.

In [ ]:
print("torch enteros   ->", torch.tensor([1, 2, 3]).dtype)
print("numpy enteros   ->", np.array([1, 2, 3]).dtype)

print("\ntorch decimales ->", torch.tensor([1.0, 2.0]).dtype)
print("numpy decimales ->", np.array([1.0, 2.0]).dtype)

PyTorch usa `float32`: la mitad de memoria y más velocidad, y para entrenar sobra.

In [ ]:
# Pedir el tipo al crear
print(torch.tensor([1, 2, 3], dtype=torch.float32).dtype,
      "|", np.array([1, 2, 3], dtype=np.float32).dtype)

# Convertir uno que ya existe
print(torch.tensor([1, 2, 3]).float().dtype,
      "|", np.array([1, 2, 3]).astype(np.float32).dtype)

----

<br>

## **Parte 4:** El puente entre los dos

Cruzar de una librería a la otra es una línea, y **las dos vistas comparten la memoria**.

In [ ]:
arreglo = np.array([1.0, 2.0, 3.0])

tensor = torch.from_numpy(arreglo)   # NumPy   -> PyTorch
de_vuelta = tensor.numpy()           # PyTorch -> NumPy

arreglo[0] = 99.0
print("el arreglo:", arreglo)
print("el tensor :", tensor, "  <- cambió sin que lo tocáramos")

Lo que llega de NumPy viene en `float64`, y la capa espera `float32`.

In [ ]:
entrada = torch.from_numpy(np.array([[3.0], [7.0]]))
print("llegó como", entrada.dtype)

try:
    capa(entrada)
except RuntimeError as error:
    print("\nRuntimeError:", error)

print("\ncon .float():", tuple(capa(entrada.float()).shape))

----

<br>

## **Parte 5:** Cambiar la forma

Los números se quedan quietos, cambia cómo se agrupan. El `-1` significa *calcúlalo tú*.

In [ ]:
print("tensor", tuple(distancias.reshape(2, 3).shape),
      "| array", distancias_np.reshape(2, 3).shape)

print("con -1:", tuple(distancias.reshape(-1).shape),
      "  | array", distancias_np.reshape(-1).shape)

Un número suelto no tiene dimensiones, y el modelo pide `(ejemplos, características)`.

In [ ]:
suelto = torch.tensor(25.0)
print("sin dimensiones:", tuple(suelto.shape))

listo = suelto.unsqueeze(0).unsqueeze(1)   # (1, 1)
print("listo          :", tuple(listo.shape), listo)

print("y de vuelta    :", listo.squeeze(), tuple(listo.squeeze().shape))

----

<br>

## **Parte 6:** Indexar y rebanar

Acá no hay nada nuevo: **la sintaxis es idéntica** a la de NumPy.

In [ ]:
predicciones = torch.tensor([[14.9], [24.1], [35.6], [45.2]])

print("t[0]    ->", predicciones[0])
print("t[:3]   ->", predicciones[:3].reshape(-1))
print("máscara ->", predicciones[predicciones > 30].reshape(-1))
print(".item() ->", predicciones[0].item(), type(predicciones[0].item()))

Con varias características, `[:, 0]` saca la primera columna.

In [ ]:
datos = torch.tensor([[3.0, 8.0, 1.0],     # distancia, hora, clima
                      [7.0, 17.0, 2.0],
                      [12.0, 12.0, 1.0]])
datos_np = datos.numpy()

print("columna 0 (tensor):", datos[:, 0])
print("columna 0 (array) :", datos_np[:, 0])

----

<br>

## **Parte 7:** Cuentas y broadcasting

`peso * distancias + sesgo` es una neurona escrita como una cuenta, sin bucles.

In [ ]:
cortas = torch.tensor([[3.0], [7.0], [12.0]])
peso, sesgo = 2.3, 8.0

print(peso * cortas + sesgo)

Cuando las formas no son iguales pero encajan, la dimensión de tamaño 1 se estira.

In [ ]:
fila = torch.tensor([[1.0, 2.0, 3.0]])          # (1, 3)
columna = torch.tensor([[4.0], [5.0], [6.0]])   # (3, 1)

print("suma:", tuple((fila + columna).shape))
print(fila + columna)

print("\nproducto de matrices:", fila @ columna)

Al resumir cambia el nombre del argumento: PyTorch dice `dim` y NumPy dice `axis`.

In [ ]:
print("torch dim=0 :", datos.sum(dim=0))
print("numpy axis=0:", datos_np.sum(axis=0))

print("\ntorch dim=1 :", datos.sum(dim=1))
print("numpy axis=1:", datos_np.sum(axis=1))

----

<br>

## **Resumen**

| Acción | NumPy | PyTorch |
| --- | --- | --- |
| Crear desde una lista | `np.array(lista)` | `torch.tensor(lista)` |
| Ceros, unos, aleatorios | `np.zeros((2, 3))` | `torch.zeros(2, 3)` |
| La forma | `a.shape` | `t.shape` |
| Cuántos números hay | `a.size` | `t.numel()` |
| Tipo por defecto con decimales | `float64` | `float32` |
| Convertir el tipo | `a.astype(np.float32)` | `t.float()` |
| Cruzar al otro lado | `t.numpy()` | `torch.from_numpy(a)` |
| Cambiar la forma | `a.reshape(-1, 1)` | `t.reshape(-1, 1)` |
| Añadir una dimensión | `np.expand_dims(a, 0)` | `t.unsqueeze(0)` |
| Quitar las de tamaño 1 | `a.squeeze()` | `t.squeeze()` |
| Indexar y rebanar | `a[:, 0]` | `t[:, 0]` |
| Sumar por columnas | `a.sum(axis=0)` | `t.sum(dim=0)` |
| Sacar un número de Python | `a.item()` | `t.item()` |

<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>